In [ ]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =====================================================================
# 1. PREPARAÇÃO DOS DADOS REUTILIZANDO A BLINDAGEM DA ETAPA 1
# =====================================================================
DATA_PATH = Path("../data/raw/Telco_customer_churn.csv")
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
df.columns = df.columns.str.strip().str.lower()

if 'total charges' in df.columns:
    df['total charges'] = pd.to_numeric(df['total charges'].astype(str).str.replace(',', '.'), errors='coerce')
    df['total charges'] = df['total charges'].fillna(0.0)

ALVO = 'churn label' if 'churn label' in df.columns else 'churn'
colunas_vazamento = ['id', 'customer id', 'churn reason', 'customer_id', 'customerid', 'churn score', 'churn value', 'cltv', ALVO]
remover_do_df = [col for col in colunas_vazamento if col in df.columns]

X_raw = df.drop(columns=remover_do_df)
y = df[ALVO].replace({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# Mapeamento dinâmico de colunas para o pipeline de cada fold
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()

# Pré-processador padrão
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

# =====================================================================
# 2. DEFINIÇÃO DOS MODELOS E VALIDAÇÃO CRUZADA (5-FOLD)
# =====================================================================
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelines = {
    "Regressão Logística (Baseline)": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest (Árvores)": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "Rede Neural Simples (MLP)": Pipeline([
        ("prep", preprocessor),
        ("model", MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42))
    ])
}

resultados_cv = []
for nome, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv_strategy, scoring=['accuracy', 'precision', 'recall', 'f1'], n_jobs=-1)
    resultados_cv.append({
        "Model": nome,
        "Accuracy": scores['test_accuracy'].mean(),
        "Precision": scores['test_precision'].mean(),
        "Recall": scores['test_recall'].mean(),
        "F1-Score": scores['test_f1'].mean()
    })

df_comparativo = pd.DataFrame(resultados_cv)
print("📋 === TABELA COMPARATIVA DE MODELOS (MÉDIA CV) ===")
for res in resultados_cv:
    print(f"\n🔹 {res['Model']}:")
    print(f"  - Acurácia Média: {res['Accuracy']:.4f} | F1-Score Médio: {res['F1-Score']:.4f}")

# =====================================================================
# 3. PIPELINE DE PRÉ-PROCESSAMENTO PRODUTIVO (ATUALIZADO SEM WARNINGS)
# =====================================================================
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_raw.select_dtypes(include=['object', 'category', 'str']).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)
melhor_res = max(resultados_cv, key=lambda k: k['F1-Score'])
print(f"\n🏆 Modelo Campeão (Maior F1): {melhor_res['Model']}")

# Treinamento final do campeão na base cheia e teste no holdout inédito
pipeline_campeao = pipelines[melhor_res['Model']]
pipeline_campeao.fit(X_train, y_train)
y_pred_final = pipeline_campeao.predict(X_test)

print(f"\n🎯 Resultados Finais do Campeão no Teste Holdout:")
print(f"  - Acurácia Final: {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  - F1-Score Final: {f1_score(y_test, y_pred_final):.4f}")

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
joblib.dump(pipeline_campeao, models_dir / "modelo_churn_campeao.joblib")
print(f"\n💾 Pipeline salvo em: {models_dir / 'modelo_churn_campeao.joblib'}")


C:\Users\selma.masuzawa\AppData\Local\Temp\ipykernel_588\3042196795.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()


📋 === TABELA COMPARATIVA DE MODELOS (MÉDIA CV) ===

🔹 Regressão Logística (Baseline):
  - Acurácia Média: 0.7982 | F1-Score Médio: 0.5820

🔹 Random Forest (Árvores):
  - Acurácia Média: 0.8016 | F1-Score Médio: 0.5384

🔹 Rede Neural Simples (MLP):
  - Acurácia Média: 0.7299 | F1-Score Médio: 0.4424

🏆 Modelo Campeão (Maior F1): Regressão Logística (Baseline)

🎯 Resultados Finais do Campeão no Teste Holdout:
  - Acurácia Final: 0.7942
  - F1-Score Final: 0.6027

💾 Pipeline salvo em: ..\models\modelo_churn_campeao.joblib
